# Laboratory Exercise 1: Job Postings

Open Jobs is a job board that lists open positions from employers and recruitment agencies across Macedonia. Its pages are a stored snapshot in `data/site`, and the setup serves them from this machine together with the statistics service of the board, so every request stays on the local loopback address. The postings are synthetic and describe no real employer, agency, or person.

Answer each request in the cells below it, and add as many cells as you need. The laboratory test asks for values and decisions that your notebook computes, so keep every result visible, run the notebook from top to bottom before you submit it, and make sure it runs on a machine without internet access.

## Setup

The cells in this section are written already. Run them before the tasks.

In [ ]:
import json
import re
import threading
from functools import partial
from http.server import SimpleHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from urllib.parse import parse_qs, urljoin, urlparse

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
from bs4 import BeautifulSoup, Tag
from playwright.async_api import async_playwright
from scipy import stats
from statsmodels.stats.multitest import multipletests

DATA = Path("data")
# Every random step draws from this generator, so a rerun gives the same result.
rng = np.random.default_rng(2026)

In [ ]:
# The course style.
PALETTE = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#1e2a24"]
GROUND, INK, GRID = "#ffffff", "#1e2a24", "#e2e6e4"
plt.rcParams.update(
    {
        "figure.figsize": (8, 4.5),
        "figure.facecolor": GROUND,
        "figure.constrained_layout.use": True,
        "savefig.facecolor": GROUND,
        "savefig.transparent": False,
        "font.family": "sans-serif",
        "font.sans-serif": ["Archivo", "DejaVu Sans"],
        "font.size": 12,
        "text.color": INK,
        "axes.facecolor": GROUND,
        "axes.edgecolor": INK,
        "axes.labelcolor": INK,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "axes.axisbelow": True,
        "axes.prop_cycle": plt.cycler(color=PALETTE),
        "grid.color": GRID,
        "xtick.color": INK,
        "ytick.color": INK,
        "legend.frameon": False,
        "image.cmap": "cividis",
    }
)
sns.set_palette(PALETTE)

In [ ]:
# The statistics service answers /api/posting-stats?page=N from a stored file,
# and every other path is a file under the site directory.
STATS = json.loads((DATA / "api" / "posting-stats.json").read_text(encoding="utf-8"))
STATS_PER_PAGE = 50


class Board(SimpleHTTPRequestHandler):
    extensions_map = {
        **SimpleHTTPRequestHandler.extensions_map,
        ".html": "text/html; charset=utf-8",
    }

    def do_GET(self) -> None:
        url = urlparse(self.path)
        if url.path != "/api/posting-stats":
            return super().do_GET()
        page = int(parse_qs(url.query).get("page", ["1"])[0])
        start = (page - 1) * STATS_PER_PAGE
        body = json.dumps(
            {
                "total": len(STATS),
                "page": page,
                "per_page": STATS_PER_PAGE,
                "results": STATS[start : start + STATS_PER_PAGE],
            }
        ).encode()
        self.send_response(200)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, format: str, *args: object) -> None:
        pass


# Port 0 lets the system choose a free port, so a second running notebook never collides.
server = ThreadingHTTPServer(("127.0.0.1", 0), partial(Board, directory=DATA / "site"))
threading.Thread(target=server.serve_forever, daemon=True).start()
BASE = f"http://127.0.0.1:{server.server_port}/"
BASE

## Task 1: Collect the Postings

The board lists its postings across several pages, and the first one is `catalogue/page-1.html` under `BASE`. Collect every posting into a table named `postings`, with one row per posting and these columns:

- `posting_id`, the identifier of the posting,
- `role`, `company`, and `city`,
- `salary` and `period`, the amount and the period it is quoted for, exactly as the posting states them,
- `experience`, the years of experience the posting requires, and `days`, the days since it was posted,
- `level`, `mode`, `contract`, and `education`, and
- `agency`, which is true when a recruitment agency posted it and false when the employer did.

A posting that does not state a field has a missing value in that column. Report how many postings the board holds and over how many pages they are spread.

In [ ]:
# Write your solution here.

## Task 2: Add the Statistics

The home page of the board, `index.html` under `BASE`, describes a statistics service. Collect every record the service holds, and add the views, the applications, and the saves of each posting to `postings`, so that every posting appears exactly once. Name the result `df`.

Report how many records the service says it holds, how many you collected, and what you did about any record that would put a posting in `df` more than once.

In [ ]:
# Write your solution here.

## Task 3: Describe the Postings

Answer each request below from `df`. Support each answer with the numbers that establish it, and with a figure wherever the request asks you to show something.

### 3.1 Column Types

Give every column of `df` a type that matches what it measures. The levels have a natural order, from junior to senior, and the type of `level` keeps it.

In [ ]:
# Write your solution here.

### 3.2 What Is Missing

Count the missing values in each column, show where they fall, and find out whether the absence of a value depends on another column.

In [ ]:
# Write your solution here.

### 3.3 Salaries in Two Periods

Some postings quote a salary per month and others per year. Compare the quoted amounts within each period, then add a column `monthly_salary` that states every salary per month.

In [ ]:
# Write your solution here.

### 3.4 The Monthly Salary

Describe the monthly salary by its centre, its spread, and the shape of its distribution, and show the distribution. Decide which summary of the centre describes a typical posting better.

In [ ]:
# Write your solution here.

### 3.5 Salary by Level

Compare the monthly salary across the levels.

In [ ]:
# Write your solution here.

### 3.6 Postings by City

Show how the postings are spread across the cities, and give the share of postings in each city.

In [ ]:
# Write your solution here.

### 3.7 Views and Applications

Show how the views and the applications of a posting move together, and measure how strongly they do. Decide whether the relationship is close to a straight line.

In [ ]:
# Write your solution here.

### 3.8 Every Pair of Numeric Columns

Show how strongly every pair of numeric columns moves together, measured by the Spearman correlation, and name the pair that moves together most strongly.

In [ ]:
# Write your solution here.

### 3.9 Salary by City and Level

Find the combination of city and level with the highest median monthly salary, counting only the combinations with at least three postings that state a salary.

In [ ]:
# Write your solution here.

## Task 4: Salaries

Use a significance level of 5 % and a confidence level of 95 % for every decision and every range in this task and the next. For each question, state the decision and give the range beside it. The questions of this task concern the postings that state a salary.

### 4.1 The Advertised Average

The home page states an average monthly salary for the postings on the board. Decide whether the average monthly salary of the postings differs from that statement, and give a range for the average monthly salary.

In [ ]:
# Write your solution here.

### 4.2 Remote and On-Site Posts

Decide whether remote and on-site postings differ in their average monthly salary, and give a range for the difference, taken as remote minus on-site.

In [ ]:
# Write your solution here.

## Task 5: Levels and Contracts

The questions of this task concern all postings.

### 5.1 Contracts by Level

Decide whether the contract type depends on the level of a posting, and give a range for the share of fixed-term postings at each level, using a percentile bootstrap.

In [ ]:
# Write your solution here.

### 5.2 The Mix of Levels

The home page states how the postings divide among the levels. Decide whether the postings match that statement, and give a range for the share of junior postings, using a percentile bootstrap.

In [ ]:
# Write your solution here.

In [ ]:
server.shutdown()